# 00. Dataset study and scope rationale

This notebook explains what the sources contain, which comparisons are feasible,
and why the project retains TVs and Samsung. It is an exploratory decision record,
not final report prose. The maintained analytical route is `00` → `01` → `02` → `03`.

**Run independently:** use this notebook's directory as the working directory.
Only `electronics_market_behavior.csv`, `appliance_brand_reviews.csv` and their
`DATA_DICTIONARY.md` are required, in `../data/01_LG_Electronics_Appliances/`.
Dependencies: Python, pandas, numpy and IPython. No other notebook, script or
previously generated dataset is read or executed. No datasets or result files are written.

Sections 3–5 compute source quality, coverage, category support, all LG/competitor
screens and review feasibility from the original CSVs. A temporary in-memory
first-view table supports scope screening; the official TV build and its browsing
variables belong to `01`. Current findings and models belong to `02` and `03`.

**Reading guide:** Sections 1–6 provide the scope argument. E1–E12 retain concise,
explicitly historical lessons from exploration; those snapshots are not re-estimated
when this notebook runs. Sections 7–11 define the handoff and reproducibility boundaries.


## 1. The question this work now supports

**Primary RQ:** How much of LG's shortfall in observed TV purchases against Samsung
at this retailer is associated with fewer sessions reaching LG's models, with the
number of models considered per reaching session, and with lower purchase conversion?

**Supporting question:** For the conversion component, how does the LG–Samsung
comparison change after accounting for initial price, calendar week and the shopper's
observed browsing situation before the model is opened?

This follows the recent reach/conversion direction in `02`, particularly Section 4.7,
and the revised report guide's Section 2. The earlier question concentrated on
view-to-cart and cart-to-purchase differences after price/month adjustment. The
exploration then raised a second issue: shoppers can arrive with different prior
browsing histories. The retained direction integrates that lesson while placing
conversion inside the broader shortfall in purchased considerations.

The decision maker is LG's team working with this retailer. The study can prioritise
an assortment, visibility or purchase-journey investigation and specify what to test
with the retailer. It cannot rank interventions by expected profit or establish that
placement, pages or checkout caused the observed patterns.

## 2. One role for each step

| Step | Question answered | Operations and outputs it owns |
| --- | --- | --- |
| **00: rationale** | Why is this scope defensible, what alternatives were considered, and what analysis follows? | Read-only source inventory and quality checks; recomputed category/competitor support, prices and concentration; review feasibility; decision rationale. No derived datasets or project-result files. |
| **01: build** | What exactly is an observation, and how are its variables constructed? | Event cleaning; first-view considerations; timestamp rules; prior browsing and outcomes; top-model split; build parameters and audits; derived CSVs. |
| **02: preliminary analysis** | What patterns does the constructed dataset show? | Category gap comparison, TV price distributions, funnel rates, shopper mix, navigation, session outcomes, reach decomposition, concentration and uncertainty tables/figures. |
| **03: models** | How do adjusted conversion, within-visit choice and considerations per model compare, and how uncertain are they? | The executable `03_models.ipynb` contains conversion LPMs with Logit checks and a Gelbach decomposition; conditional Logit among single-purchase visits; model-level Poisson counts; an optional within-model price extension. Inputs are the saved `01` datasets; outputs are Tables 5.1–5.8, Figure 5.1 and an exhibit index under `results/models/`. The notebook is the maintained modelling source. |

The logical sequence is **scope → measurement → description → modelling**. The
technical build starts from the raw behaviour file: `01` needs no output from `00`.
The scope table exists only in memory and is never an input to later steps. `00` does not import or execute the other scripts and never requires `01`
or `02` to have been run.


In [1]:
# 01. Load the original sources for scope screening only.
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.max_colwidth', None)

DATA_DIR = Path('..') / 'data' / '01_LG_Electronics_Appliances'
required = ['electronics_market_behavior.csv', 'appliance_brand_reviews.csv', 'DATA_DICTIONARY.md']
missing = [name for name in required if not (DATA_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f'Run from Marketing-Project and check {DATA_DIR}: {missing}')

behavior = pd.read_csv(DATA_DIR / required[0])
reviews = pd.read_csv(DATA_DIR / required[1])


## 3. What the available sources can answer

The local `DATA_DICTIONARY.md` is the source for the following data meanings and
sampling statements; the next cells check coverage in the actual CSVs.

| Source | Observation and useful information | Questions it can support | Boundary |
| --- | --- | --- | --- |
| Behaviour | A product event with user, session, product, brand, category, time and listed price | Observed consideration and purchase journeys; relative reach within the retained brands; association of browsing and conversion | Historical retailer proxy; no product specifications, stock, traffic source, advertising exposure, costs or intervention assignment |
| Reviews | A review with rating, text, date and verification; repeated product listing fields | Satisfaction and review-topic comparisons within represented appliance categories | Self-selected reviewers, different dates and product mix; listing fields are not measured at purchase |

The instructor retained LG and five competitors in each behaviour category and
sampled whole sessions within the curated scope. This does not provide full user
histories, all retailer traffic or population brand shares. Review counts are capped
within brand/category cells and are not a popularity measure. Neither file is LG's
proprietary customer database. The retailer, country and behaviour-price currency
are unidentified; the review listing price is described as USD.

In [2]:
# 02. Check source compatibility without creating an analytical cohort.
required_behavior = {'event_type', 'user_id', 'session_id', 'product_id', 'brand', 'product_category', 'price', 'event_time'}
required_reviews = {'review_id', 'product_id', 'brand', 'product_category', 'review_rating', 'review_text', 'review_date', 'verified_purchase'}
assert required_behavior.issubset(behavior.columns)
assert required_reviews.issubset(reviews.columns)

compatibility = pd.Series({
    'Behaviour categories': ', '.join(sorted(behavior['product_category'].dropna().unique())),
    'Review categories': ', '.join(sorted(reviews['product_category'].dropna().unique())),
    'Shared product-ID labels': len(set(behavior['product_id'].dropna()) & set(reviews['product_id'].dropna())),
    'User ID available in reviews': 'user_id' in reviews.columns,
}, name='Source compatibility')
display(compatibility.to_frame())


# Read-only quality audit of the behaviour source, before scope aggregation.
field_meanings = {
    'event_time': 'UTC event timestamp', 'event_type': 'view, cart or purchase',
    'user_id': 'Observed user identifier; history may be incomplete',
    'session_id': 'Session identifier; combine with user_id',
    'product_id': 'Product identifier, not a specification match',
    'brand': 'Recorded brand', 'product_category': 'Recorded category',
    'price': 'Price recorded at this event; currency unidentified',
}
behavior_schema = pd.DataFrame({
    'Meaning': pd.Series(field_meanings), 'dtype': behavior.dtypes.astype(str),
    'Missing rows': behavior.isna().sum(), 'Distinct values': behavior.nunique(),
})
display(behavior_schema)

# Match the official build's order: remove exact raw duplicates, then normalise.
screen_events = behavior.drop_duplicates().copy()
screen_events['event_time'] = pd.to_datetime(screen_events['event_time'], utc=True, errors='coerce')
for column in ['event_type', 'brand', 'product_category']:
    screen_events[column] = screen_events[column].str.strip().str.lower()
screen_events['price'] = pd.to_numeric(screen_events['price'], errors='coerce')
invalid = (screen_events.isna().any(axis=1)
           | ~np.isfinite(screen_events['price']) | screen_events['price'].le(0)
           | ~screen_events['event_type'].isin(['view', 'cart', 'purchase']))
behavior_audit = pd.Series({
    'Raw event rows': len(behavior),
    'Exact duplicate rows excluded from screening': len(behavior) - len(screen_events),
    'Distinct event rows after deduplication': len(screen_events),
    'Invalid rows requiring investigation': int(invalid.sum()),
    'Session IDs shared by several users': int((screen_events.groupby('session_id')['user_id'].nunique() > 1).sum()),
}, name='Count')
display(behavior_audit.to_frame())
assert not invalid.any(), 'Resolve invalid source rows before interpreting scope screens.'
assert screen_events.groupby('product_id')[['brand', 'product_category']].nunique().eq(1).all().all()
assert len(behavior) == len(screen_events) + behavior.duplicated().sum()
display(pd.Series({'First event (UTC)': screen_events['event_time'].min(),
                   'Last event (UTC)': screen_events['event_time'].max()}, name='Observed window').to_frame())


,Source compatibility
Behaviour categories,"microwave, monitor, refrigerator, tv, vacuum_cleaner, washing_machine"
Review categories,"air_conditioner, cooking, dishwasher, laundry, other_appliance, refrigerator_freezer"
Shared product-ID labels,0
User ID available in reviews,False


,Meaning,dtype,Missing rows,Distinct values
brand,Recorded brand,str,0,21
event_time,UTC event timestamp,str,0,569714
event_type,"view, cart or purchase",str,0,3
price,Price recorded at this event; currency unidentified,float64,0,6671
product_category,Recorded category,str,0,6
product_id,"Product identifier, not a specification match",str,0,1613
session_id,Session identifier; combine with user_id,str,0,162702
user_id,Observed user identifier; history may be incomplete,str,0,138863


,Count
Raw event rows,649605
Exact duplicate rows excluded from screening,1247
Distinct event rows after deduplication,648358
Invalid rows requiring investigation,0
Session IDs shared by several users,4


,Observed window
First event (UTC),2019-10-01 00:06:07+00:00
Last event (UTC),2019-11-30 23:56:30+00:00


The dictionary explicitly states that the products and users differ and that the
files have no join relationship. An accidental matching ID would not establish a
valid link. Reviews have no TV category, and similarly named appliance categories
are not a product crosswalk. Brand-level averages from reviews cannot be attached
to individual behaviour rows to explain their purchases.

**Decision:** use behaviour for this RQ. Reviews remain a credible *alternative
project*, not an extra explanatory variable for TV conversion. They are not rejected
merely because one column has many missing values.

## 4. Coverage and comparable support before selecting the scope

The tables below recompute coverage for every observed category and brand. Event
counts describe activity; viewed considerations describe comparable opportunities
within a user-session-product key. Neither measures market share or physical units sold.
The original CSV is unchanged. Invalid records or mappings stop the screen for investigation.

The catalogue matrix counts products appearing in any event, including products
without a view. A zero denotes missing sample coverage, not absence from the market.
The category/event and category/month tables show how the observed sample is composed.


In [3]:
# Catalogue and activity coverage, all categories and brands.
catalogue = screen_events[['product_id', 'brand', 'product_category']].drop_duplicates()
coverage = pd.crosstab(catalogue['brand'], catalogue['product_category'])
display(coverage)
category_activity = screen_events.groupby('product_category').agg(
    events=('event_type', 'size'), users=('user_id', 'nunique'),
    products=('product_id', 'nunique'), brands=('brand', 'nunique'),
)
display(category_activity)
display(pd.crosstab(screen_events['product_category'], screen_events['event_type']))
display(pd.crosstab(screen_events['product_category'], screen_events['event_time'].dt.strftime('%Y-%m')))
print('Distinct sampled brands:', len(coverage))
print('Categories containing both LG and Samsung:',
      ', '.join(coverage.columns[coverage.reindex(['lg', 'samsung'], fill_value=0).gt(0).all(axis=0)]))


Distinct sampled brands: 21
Categories containing both LG and Samsung: microwave, monitor, refrigerator, tv, vacuum_cleaner, washing_machine


product_category,microwave,monitor,refrigerator,tv,vacuum_cleaner,washing_machine
brand,,,,,,
acer,0,100,0,0,0,0
aoc,0,54,0,0,0,0
arg,4,0,0,0,0,0
artel,0,0,0,38,0,29
atlant,0,0,109,0,0,0
beko,0,0,30,0,0,23
bosch,0,0,0,0,54,0
dauscher,10,0,0,0,0,0
elenberg,5,0,0,0,0,0


,events,users,products,brands
product_category,,,,
microwave,26569,8716,76,6
monitor,21718,4778,365,6
refrigerator,108259,26554,387,6
tv,211958,50373,290,6
vacuum_cleaner,117983,27551,253,6
washing_machine,161871,35711,242,6


event_type,cart,purchase,view
product_category,,,
microwave,1228,566,24775
monitor,519,192,21007
refrigerator,3339,1236,103684
tv,9706,3785,198467
vacuum_cleaner,4706,1859,111418
washing_machine,6442,2730,152699


event_time,2019-10,2019-11
product_category,,
microwave,9520,17049
monitor,7747,13971
refrigerator,40987,67272
tv,72081,139877
vacuum_cleaner,39705,78278
washing_machine,62535,99336


### 4.1 Why TVs, with washing machines as a serious alternative

The next table uses one **user × session × product with at least one view**.
Its price is the first-view price. A cart or purchase qualifies at or after that
view in the same key, including equal timestamps, consistently with `01`.
A purchase after cart must be strictly later than the first eligible cart.
Counts of purchased considerations do not certify the number of units bought.

For each category and brand we compute support, price quartiles and concentration.
The top-five share is the percentage of considerations belonging to the five
most-viewed products in that category/brand. It is distinct from `01`'s 4% top-model
rule. Categories with fewer than five viewed products can have a 100% share.
The screen gives every consideration one price observation; products attracting
more considerations therefore contribute more to its price distribution.

The comparison balances support, price proximity, concentration and a focused
managerial question. It does not rank categories by significance or use an invented
selection score. This choice followed exploration of the available data.


In [4]:
# Temporary scope table; no browsing variables and no dataset export.
SCREEN_KEY = ['user_id', 'session_id', 'product_id']
screen_views = screen_events.loc[screen_events['event_type'].eq('view')]
scope = (screen_views.sort_values('event_time', kind='stable').drop_duplicates(SCREEN_KEY)
         [SCREEN_KEY + ['brand', 'product_category', 'event_time', 'price']]
         .rename(columns={'event_time': 'first_view', 'price': 'initial_price'}))
screen_purchase = (screen_events.loc[screen_events['event_type'].eq('purchase')]
                   .groupby(SCREEN_KEY)['event_time'].max().rename('last_purchase'))
screen_cart = screen_events.loc[screen_events['event_type'].eq('cart'), SCREEN_KEY + ['event_time']].merge(
    scope[SCREEN_KEY + ['first_view']], on=SCREEN_KEY, validate='many_to_one')
screen_cart = screen_cart.loc[screen_cart['event_time'].ge(screen_cart['first_view'])]
first_cart = screen_cart.groupby(SCREEN_KEY)['event_time'].min().rename('first_cart')
scope = scope.join(screen_purchase, on=SCREEN_KEY).join(first_cart, on=SCREEN_KEY)
scope['purchased'] = scope['last_purchase'].ge(scope['first_view']).astype(int)
scope['has_cart'] = scope['first_cart'].notna().astype(int)
scope['purchased_after_cart'] = scope['last_purchase'].gt(scope['first_cart']).astype(int)
assert not scope.duplicated(SCREEN_KEY).any()
assert scope['purchased_after_cart'].le(scope['has_cart']).all()

screen_by = ['product_category', 'brand']
scope_support = scope.groupby(screen_by).agg(
    considerations=('purchased', 'size'), users=('user_id', 'nunique'),
    viewed_products=('product_id', 'nunique'), purchases=('purchased', 'sum'),
    carts=('has_cart', 'sum'), purchases_after_cart=('purchased_after_cart', 'sum'),
)
quartiles = scope.groupby(screen_by)['initial_price'].quantile([0.25, 0.5, 0.75]).unstack()
quartiles.columns = ['price_q25', 'median_price', 'price_q75']
scope_support = scope_support.join(quartiles)
product_counts = scope.groupby(screen_by + ['product_id']).size().rename('considerations').reset_index()
ranked = product_counts.sort_values(screen_by + ['considerations', 'product_id'],
                                   ascending=[True, True, False, True], kind='stable')
top_five = ranked.groupby(screen_by).head(5).groupby(screen_by)['considerations'].sum()
scope_support['top5_share_pct'] = 100 * top_five / scope_support['considerations']
scope_support['no_cart'] = scope_support['considerations'] - scope_support['carts']
scope_support['cart_without_later_purchase'] = scope_support['carts'] - scope_support['purchases_after_cart']
assert scope_support['considerations'].sum() == len(scope)
display(scope_support.round(2))

lg_support = scope_support.xs('lg', level='brand').add_suffix('_LG')
screen_candidates = (scope_support.reset_index().query("brand != 'lg'")
                     .set_index('product_category').add_suffix('_competitor').join(lg_support, how='inner')
                     .reset_index().rename(columns={'brand_competitor': 'competitor'}))
for metric in ['considerations', 'purchases']:
    screen_candidates['combined_' + metric] = (screen_candidates[metric + '_LG']
                                               + screen_candidates[metric + '_competitor'])
screen_candidates['median_price_distance_pct'] = 100 * (
    screen_candidates['median_price_competitor'] / screen_candidates['median_price_LG'] - 1).abs()
screen_candidates['iqr_overlap_width'] = (
    screen_candidates[['price_q75_LG', 'price_q75_competitor']].min(axis=1)
    - screen_candidates[['price_q25_LG', 'price_q25_competitor']].max(axis=1)).clip(lower=0)
assert not screen_candidates.duplicated(['product_category', 'competitor']).any()

category_shortlist = screen_candidates.loc[screen_candidates['competitor'].eq('samsung'), [
    'product_category', 'considerations_LG', 'considerations_competitor', 'purchases_LG',
    'purchases_competitor', 'combined_considerations', 'combined_purchases',
    'viewed_products_LG', 'viewed_products_competitor', 'median_price_LG', 'median_price_competitor',
    'top5_share_pct_LG', 'top5_share_pct_competitor',
]].set_index('product_category')
display(category_shortlist.round(2))
print(f'{len(scope):,} viewed considerations across {len(scope_support)} category/brand cells; '
      f'{len(screen_candidates)} LG/competitor comparisons.')


363,277 viewed considerations across 36 category/brand cells; 30 LG/competitor comparisons.


considerations  users  viewed_products  purchases  \
product_category brand                                                         
microwave        arg                 3427   2934                4        160   
                 dauscher            1773   1395               10         33   
                 elenberg            1320   1227                5         93   
                 lg                  2822   2186               17         71   
                 midea               1523   1292               11         44   
                 samsung             4701   3045               29        150   
monitor          acer                4614   2422              100         66   
                 aoc                 1099    690               54          6   
                 hp                  1597   1093               82         24   
                 lg                  1312    847               40         13   
                 qmax                 560    422               28          9   
                 samsung             3718   2196               61         59   
refrigerator     atlant              9219   5411              109        142   
                 beko                9794   6412               30        210   
                 indesit            17715   9659               70        363   
                 lg                 13079   7984               49        191   
                 midea               5863   3868               88         88   
                 samsung             9950   5703               41        132   
tv               artel              19872  14297               38        624   
                 haier               9654   7373               20        340   
                 kivi                6465   5087               19        222   
                 lg                 22213  12908              105        505   
                 samsung            52499  29327               91       1645   
                 tcl                 5008   3838               17        156   
vacuum_cleaner   bosch              10649   7058               54        306   
                 lg                  5493   3376               27        132   
                 samsung            26308  14546               73        906   
                 tefal               6170   4341               49        133   
                 vitek               5628   3867               26        137   
                 xiaomi              8261   5496               24        139   
washing_machine  artel               4116   2112               29        120   
                 beko                8967   6113               23        285   
                 indesit            16803   8940               45        412   
                 lg                 21959  11546               69        544   
                 midea               6442   4359               28        265   
                 samsung            32684  17891               48        910   

                           carts  purchases_after_cart  price_q25  \
product_category brand                                              
microwave        arg         269                   115      56.60   
                 dauscher     59                    19      57.89   
                 elenberg     93                    42      50.91   
                 lg          155                    54      89.38   
                 midea        70                    29      48.65   
                 samsung     275                   113     113.77   
monitor          acer        127                    40     126.88   
                 aoc          14                     5     137.18   
                 hp           53                    13     135.65   
                 lg           37                     9     125.82   
                 qmax         13                     5      67.40   
                 samsung     140                    42     132.14   
refrigerator     atl

,considerations_LG,considerations_competitor,purchases_LG,purchases_competitor,combined_considerations,combined_purchases,viewed_products_LG,viewed_products_competitor,median_price_LG,median_price_competitor,top5_share_pct_LG,top5_share_pct_competitor
product_category,,,,,,,,,,,,
microwave,2822,4701,71,150,7523,221,17,29,94.96,125.87,89.40,62.69
monitor,1312,3718,13,59,5030,72,40,61,221.32,223.63,31.86,48.52
refrigerator,13079,9950,191,132,23029,323,49,41,535.12,643.26,59.77,49.88
tv,22213,52499,505,1645,74712,2150,105,91,460.84,493.96,42.31,52.38
vacuum_cleaner,5493,26308,132,906,31801,1038,27,73,172.38,92.64,60.00,50.44
washing_machine,21959,32684,544,910,54643,1454,69,48,380.96,347.47,51.16,58.40


### 4.2 Why Samsung rather than the other sampled TV brands

Each observed competitor is compared with LG **within its category**, using the
same first-view sample as Section 4.1. The complete table includes every candidate;
the second table isolates TVs for readability.

- **Median-price distance (%)** = `100 × abs(rival median / LG median − 1)`.
- **Central price range** = the interquartile range, from Q25 to Q75.
- **Overlap width** = `max(0, min(Q75_LG, Q75_rival) − max(Q25_LG, Q25_rival))`,
  expressed in the unidentified price units. Zero means no positive-width overlap
  of these central ranges, not no overlap anywhere in the two distributions.
- Support and concentration use considered products and considerations, not all
  listed products equally weighted. These screens do not match specifications.

Price proximity is one criterion alongside sample support and assortment. The
historical scope choice remains exploratory; these recalculations make its evidence
inspectable without prescribing a mechanical winner rule.


In [5]:
# Show all candidates, then the TV shortlist, without filtering for a preferred result.
screen_columns = ['product_category', 'competitor', 'considerations_LG', 'considerations_competitor',
                  'purchases_LG', 'purchases_competitor', 'viewed_products_LG', 'viewed_products_competitor',
                  'median_price_LG', 'median_price_competitor', 'median_price_distance_pct',
                  'iqr_overlap_width', 'combined_considerations', 'combined_purchases',
                  'top5_share_pct_LG', 'top5_share_pct_competitor']
ordered_candidates = screen_candidates.sort_values(['product_category', 'median_price_distance_pct', 'competitor'])
display(ordered_candidates[screen_columns].set_index(['product_category', 'competitor']).round(2))
tv_candidates = ordered_candidates.loc[ordered_candidates['product_category'].eq('tv')]
display(tv_candidates[screen_columns].drop(columns='product_category').set_index('competitor').round(2))
closest = ordered_candidates.groupby('product_category', sort=True).head(1)
print('Smallest median-price distance in each category (descriptive screen):')
display(closest[['product_category', 'competitor', 'median_price_distance_pct']].set_index('product_category').round(2))


Smallest median-price distance in each category (descriptive screen):


considerations_LG  considerations_competitor  \
product_category competitor                                                 
microwave        samsung                  2822                       4701   
                 arg                      2822                       3427   
                 dauscher                 2822                       1773   
                 elenberg                 2822                       1320   
                 midea                    2822                       1523   
monitor          samsung                  1312                       3718   
                 hp                       1312                       1597   
                 aoc                      1312                       1099   
                 acer                     1312                       4614   
                 qmax                     1312                        560   
refrigerator     samsung                 13079                       9950   
                 indesit                 13079                      17715   
                 atlant                  13079                       9219   
                 beko                    13079                       9794   
                 midea                   13079                       5863   
tv               samsung                 22213                      52499   
                 tcl                     22213                       5008   
                 haier                   22213                       9654   
                 kivi                    22213                       6465   
                 artel                   22213                      19872   
vacuum_cleaner   tefal                    5493                       6170   
                 bosch                    5493                      10649   
                 samsung                  5493                      26308   
                 vitek                    5493                       5628   
                 xiaomi                   5493                       8261   
washing_machine  samsung                 21959                      32684   
                 beko                    21959                       8967   
                 indesit                 21959                      16803   
                 midea                   21959                       6442   
                 artel                   21959                       4116   

                             purchases_LG  purchases_competitor  \
product_category competitor                                       
microwave        samsung               71                   150   
                 arg                   71                   160   
                 dauscher              71                    33   
                 elenberg              71                    93   
                 midea                 71                    44   
monitor          samsung               13                    59   
                 hp                    13                    24   
                 aoc                   13                     6   
                 acer                  13                    66   
                 qmax                  13                     9   
refrigerator     samsung              191                   132   
                 indesit              191                   363   
                 atlant               191                   142   
                 beko                 191                   210   
                 midea                191                    88   
tv               samsung              505                  1645   
                 tcl                  505                   156   
                 haier                505                   340   
                 kivi                 505                   222   
                 artel                505                   624   
vacuum_cleaner   tefal                132                   133   
                 bosch 

,considerations_LG,considerations_competitor,purchases_LG,purchases_competitor,viewed_products_LG,viewed_products_competitor,median_price_LG,median_price_competitor,median_price_distance_pct,iqr_overlap_width,combined_considerations,combined_purchases,top5_share_pct_LG,top5_share_pct_competitor
competitor,,,,,,,,,,,,,,
samsung,22213,52499,505,1645,105,91,460.84,493.96,7.19,244.39,74712,2150,42.31,52.38
tcl,22213,5008,505,156,105,17,460.84,251.95,45.33,0.00,27221,661,42.31,70.75
haier,22213,9654,505,340,105,20,460.84,231.64,49.74,0.00,31867,845,42.31,72.25
kivi,22213,6465,505,222,105,19,460.84,192.93,58.14,0.00,28678,727,42.31,73.78
artel,22213,19872,505,624,105,38,460.84,154.19,66.54,0.00,42085,1129,42.31,83.89


,competitor,median_price_distance_pct
product_category,,
microwave,samsung,32.55
monitor,samsung,1.04
refrigerator,samsung,20.21
tv,samsung,7.19
vacuum_cleaner,tefal,19.38
washing_machine,samsung,8.79


The recomputed coverage and first-view-price screen support Samsung as a feasible
benchmark. They do not establish that it is the closest competitor for every
individual TV, or require it to rank first in every other category.

Samsung is therefore a feasible primary benchmark within this sample. This does
not establish matched products, identical customer intent or market leadership.
The other TV brands remain in the `01` dataset as contextual alternatives and prior
browsing information; they are not discarded from observed journeys. The group
`other_four` is a convenience, not evidence that its brands share a positioning.

An alternative is to compare LG with all remaining brands pooled.
That benchmark changes when the competitors' price and product mix changes, so it
is useful as context rather than a substitute for a clearly named comparator.


**Category decision:** retain TVs for the combined LG/Samsung sample, feasible
price benchmark and focused journey question. Washing machines remain a credible
alternative, with more LG purchases in this sample. TVs need not maximise every
criterion. Current conversion comparisons and uncertainty are developed in `02`;
these screening tables establish feasibility rather than a significant brand effect.


## 5. Review-based alternatives: feasible, but a different project

This section owns the review feasibility exploration, which is
not performed in the TV build or preliminary analysis beyond noting the absence
of TVs. It examines whether a satisfaction question was a real alternative.

In [6]:
# 04. Check review quality and retain ratings/text despite missing listing prices.
review_quality = pd.DataFrame({
    'dtype': reviews.dtypes.astype(str),
    'missing_rows': reviews.isna().sum(),
    'missing_pct': 100 * reviews.isna().mean(),
    'distinct_values': reviews.nunique(),
})
display(review_quality.round(2))
review_dates = pd.to_datetime(reviews['review_date'], errors='coerce')
ratings = pd.to_numeric(reviews['review_rating'], errors='coerce')
prices = pd.to_numeric(reviews['price'], errors='coerce')
text_available = reviews['review_text'].fillna('').astype(str).str.strip().ne('')
review_checks = pd.Series({
    'Exact duplicate review rows': int(reviews.duplicated().sum()),
    'Duplicate review IDs': int(reviews['review_id'].duplicated().sum()),
    'Unparseable or missing review dates': int(review_dates.isna().sum()),
    'Ratings outside the five-star scale or missing': int((~ratings.isin([1, 2, 3, 4, 5])).sum()),
    'Missing or blank review text': int((~text_available).sum()),
    'Usable positive listing prices (%)': 100 * (np.isfinite(prices) & prices.gt(0)).mean(),
})
display(review_checks.to_frame('Value').round(2))
assert not reviews['review_id'].duplicated().any()
assert review_dates.notna().all() and ratings.isin([1, 2, 3, 4, 5]).all()
assert reviews[['product_id', 'brand', 'product_category']].notna().all().all()
assert reviews.groupby('product_id')[['brand', 'product_category']].nunique().eq(1).all().all()


,dtype,missing_rows,missing_pct,distinct_values
review_id,str,0,0.00,20004
product_id,str,0,0.00,2750
brand,str,0,0.00,8
product_category,str,0,0.00,6
product_title,str,0,0.00,2637
price,float64,15883,79.40,141
product_average_rating,float64,0,0.00,40
product_rating_count,int64,0,0.00,167
review_rating,float64,0,0.00,5
review_title,str,2,0.01,16608


,Value
Exact duplicate review rows,0.0
Duplicate review IDs,0.0
Unparseable or missing review dates,0.0
Ratings outside the five-star scale or missing,0.0
Missing or blank review text,21.0
Usable positive listing prices (%),20.6


Missing price prevents some price-based analyses; it does not invalidate an observed
rating or usable text. No price is imputed and no review is removed because its
listing price is unavailable. `review_rating` is a row-level response;
`product_average_rating` and `product_rating_count` are repeated listing-level
information and must not be counted as independent reviewer outcomes.

In [7]:
# 05. Map category support and the observation window for a satisfaction study.
review_work = reviews.assign(
    brand=reviews['brand'].astype('string').str.strip().str.lower(),
    observed_date=review_dates,
    rating=ratings,
    usable_price=np.isfinite(prices) & prices.gt(0),
    usable_text=text_available,
)
review_coverage = review_work.groupby(['brand', 'product_category']).agg(
    reviews=('review_id', 'size'), products=('product_id', 'nunique'),
).unstack('product_category', fill_value=0)
display(review_coverage)
review_support = review_work.groupby('product_category').agg(
    reviews=('review_id', 'size'), products=('product_id', 'nunique'),
    brands=('brand', 'nunique'), first_review=('observed_date', 'min'),
    last_review=('observed_date', 'max'), usable_price_share=('usable_price', 'mean'),
    usable_text_share=('usable_text', 'mean'),
)
review_support[['usable_price_share', 'usable_text_share']] *= 100
display(review_support.rename(columns={
    'usable_price_share': 'usable_price_pct', 'usable_text_share': 'usable_text_pct',
}).round({'usable_price_pct': 2, 'usable_text_pct': 2}))


reviews                                             \
product_category air_conditioner cooking dishwasher laundry other_appliance   
brand                                                                         
bosch                          0     205        611     204               0   
electrolux                     0     153         63     230               8   
frigidaire                     7    1287        446     263            1287   
ge                            16    1287        330     616            1287   
haier                         10      14          0    1287             130   
lg                             0     327         88    1287               0   
samsung                        0     568        210     613               6   
whirlpool                      0     272        211     806              41   

                                             products                     \
product_category refrigerator_freezer air_conditioner cooking dishwasher   
brand                                                                      
bosch                             150               0      71         72   
electrolux                        112               0      42          7   
frigidaire                       1287               1     210         81   
ge                                939               2     279         87   
haier                             422               2       6          0   
lg                               1221               0      57         13   
samsung                          1287               0      81         26   
whirlpool                         416               0      81         56   

                                                               
product_category laundry other_appliance refrigerator_freezer  
brand                                                          
bosch                 31               0                   16  
electrolux            53               3                   45  
frigidaire            45              19                  112  
ge                   111              12                  195  
haier                 21              12                   32  
lg                   186               0                  165  
samsung              138               2                  166  
whirlpool            136               7                   69

,reviews,products,brands,first_review,last_review,usable_price_pct,usable_text_pct
product_category,,,,,,,
air_conditioner,33,5,3,2017-09-26,2022-05-30,21.21,100.00
cooking,4113,827,8,2006-08-30,2023-08-08,15.78,99.85
dishwasher,1959,342,7,2006-07-30,2023-07-24,4.24,100.00
laundry,5306,721,8,2005-04-15,2023-08-23,4.05,99.91
other_appliance,2759,55,6,2004-07-12,2023-09-01,57.05,99.78
refrigerator_freezer,5834,800,8,2005-09-12,2023-08-18,27.31,99.93


In [8]:
# 06. Describe ratings and verification without treating either as product quality.
display(pd.crosstab(review_work['brand'], review_work['rating']).rename_axis(columns='Review stars'))
verified_text = reviews['verified_purchase'].astype(str).str.strip().str.lower()
assert verified_text.isin(['true', 'false', '1', '0']).all(), 'Check the source verification encoding.'
review_work['verified'] = verified_text.isin(['true', '1'])
display(pd.crosstab(review_work['brand'], review_work['verified']).rename(
    columns={False: 'Unverified reviews', True: 'Verified reviews'}))

review_candidates = review_work[review_work['brand'].isin(['lg', 'samsung'])].groupby(
    ['product_category', 'brand']
).agg(reviews=('review_id', 'size'), products=('product_id', 'nunique'),
      mean_rating=('rating', 'mean'), verified_share=('verified', 'mean'),
      first_review=('observed_date', 'min'), last_review=('observed_date', 'max'))
review_candidates['verified_share'] *= 100
assert set(review_candidates.index.get_level_values('brand')) == {'lg', 'samsung'}
display(review_candidates.rename(columns={'verified_share': 'verified_pct'}).round(
    {'mean_rating': 2, 'verified_pct': 2}))


Review stars,1.0,2.0,3.0,4.0,5.0
brand,,,,,
bosch,527,131,73,111,328
electrolux,304,63,23,46,130
frigidaire,1253,290,335,497,2202
ge,1658,384,269,434,1730
haier,258,87,127,285,1106
lg,1229,216,170,282,1026
samsung,1601,267,136,194,486
whirlpool,968,141,87,160,390


verified,Unverified reviews,Verified reviews
brand,,
bosch,887,283
electrolux,411,155
frigidaire,806,3771
ge,1904,2571
haier,490,1373
lg,1901,1022
samsung,2270,414
whirlpool,1307,439


reviews  products  mean_rating  verified_pct  \
product_category     brand                                                   
cooking              lg           327        57         3.31         54.13   
                     samsung      568        81         2.46         23.77   
dishwasher           lg            88        13         1.64          1.14   
                     samsung      210        26         1.50          4.76   
laundry              lg          1287       186         2.91         27.89   
                     samsung      613       138         2.07         16.15   
other_appliance      samsung        6         2         2.33         66.67   
refrigerator_freezer lg          1221       165         2.84         39.72   
                     samsung     1287       166         2.14         12.90   

                             first_review last_review  
product_category     brand                             
cooking              lg        2006-12-12  2023-05-21  
                     samsung   2009-09-09  2023-07-25  
dishwasher           lg        2006-07-30  2023-07-12  
                     samsung   2010-07-01  2023-07-08  
laundry              lg        2006-11-09  2023-08-12  
                     samsung   2009-08-07  2023-07-10  
other_appliance      samsung   2022-04-28  2023-05-23  
refrigerator_freezer lg        2008-03-03  2023-07-07  
                     samsung   2008-08-19  2023-06-26

The rating and verification tables reproduce the *questions raised* by the old
review exploration: do raw brand patterns persist within the same categories,
periods, products and verification groups? Unequal extremes in the rating
distribution are not population defect rates. Unverified does not mean fake, and
verified does not prove an unbiased review. Long time coverage is not a balanced
panel; dates, sampled products and review selection differ.

A separate satisfaction project could compare ratings within overlapping appliance
categories and then inspect text themes, with product dependence and verified-only
checks. It would need its own cleaning/measurement decisions and an appropriate
course-method justification. These descriptive tables do not establish adjusted
brand quality differences or explain the retailer's TV purchase gap.

## 6. Research directions considered and reasons for the chosen route

| Direction considered | Real merit | Why it is not the primary question here | What survives |
| --- | --- | --- | --- |
| Two stage gaps: view→cart and cart→purchase | Localises observed progression; motivated the initial stage models | Different denominators; cart entry selects shoppers; omits the reach component of the purchase-count shortfall | Diagnostic funnel in `02`; conditional stage models only if needed later |
| Purchase conversion adjusted only for price and month | One binary commercial endpoint; simple course Logit | Leaves observed browsing composition and the count/reach distinction unexamined | A baseline specification, not the final explanation |
| Purchase gap after prior cross-brand browsing | Uses information available before the consideration; follows the shopper-mix exploration | A changed adjusted estimate does not establish a cause or a profitable targeting action | Supporting question and a timed browsing specification |
| Whole-period single-brand versus cross-brand shoppers | Exposes that brands can attract different observed browsing profiles | Uses later behaviour and incomplete histories; group membership is selected | Descriptive lesson and optional retrospective sensitivity, not a first-view predictor |
| Washing-machine conversion | Credible sample and competitor alternative | A different scope, without a demonstrated reason to replace the retained TV investigation | Explicit alternative in the scope rationale |
| Other behaviour categories or all competitors pooled | Tests whether the TV/Samsung comparison was arbitrary | Support, price comparability, assortment and purchase processes vary | All-category/competitor screening; category comparisons in `02` |
| Appliance satisfaction and review topics | Usable ratings/text in overlapping LG/Samsung appliance categories | Different products, users, periods and managerial outcome; no TV linkage | Review feasibility evidence above |
| Segmentation/targeting, RFM or K-means | Could organise observed browsing if a targeting decision required it | Adds a new objective; no demographics or complete customer histories; clusters do not explain causes | Simple observed browsing variables instead of a new primary project |
| Pricing elasticity, optimal discounts or ROI | Direct economic decision | No exogenous price variation, complete demand exposure, costs or margins; listed prices also vary with product mix | Initial price as an associational adjustment, not an optimisation tool |

**Retained route:** explain the observed purchase-count shortfall with an accounting
breakdown, inspect where the descriptive differences sit, then assess the conversion
comparison with a transparent purchase model. The alternatives are excluded for
fit to this decision and data, not because their estimated gaps or p-values were
less attractive. This does not certify TV as the only defensible research project.


## Evidence record: what the exploratory work actually found

This record is readable without opening the retired notebooks or their code. Each
block states the question, the observed result, its limit and the decision it informed.
The historical analyses were **re-executed on 18 November 2026 against the original
local CSVs** before these summaries were generated. Numerical values below are
**fixed historical snapshots**, not results recomputed by the current screening cells.
Selected aggregate tables are embedded below. More detailed snapshots are retained
in notebook metadata as an archive, not as executable calculations or runtime inputs.

The historical consideration definition required cart/purchase events **strictly
later** than first view. `01` now accepts same-second carts/purchases. Raw event
counts also differ before/after deduplication. Whole-period shopper types and
user-only uncertainty in the old experiments differ from the current pipeline.
Therefore these numbers explain **how the project developed**, not what the final
report must reproduce. Current descriptive results belong to `02`, models to `03`.
Sections 3–5 now recompute the scope evidence using the current equal-timestamp
convention. The snapshots below preserve the earlier definitions and their lessons.
Historical model diagnostics are not a second set of current report results.


### E1. Were both sources usable, and could they be combined?

**Question and method.** Inventory schemas, missingness, dates, duplicate rows,
identifier mappings and category coverage before choosing an outcome.

**Result.** Behaviour contained 649,605 events, eight fields, 21 brands and 1,613
products over October–November 2019. Removing 1,247 exact copies left 648,358 events;
no invalid rows were quarantined under the exploratory rules. Four session IDs
were shared across users, while product-to-brand/category mappings were consistent.
Reviews contained 20,004 rows, 14 fields and 2,750 products across eight brands,
from July 2004 to September 2023; no duplicates/invalid reviews were removed.
Listing price was missing in 79.4%, while ratings and verification were complete.
There were six missing text values; the more explicit current blank-string check
also finds whitespace-only content, giving 21 missing-or-blank texts.

**Conclusion and decision.** Use the composite user/session key and an audited
build, rather than count events as independent purchases or people. Reviews have
no TV category and no valid product/user join to behaviour; use them only for a
separate satisfaction question. Missing listing prices do not make ratings/text
unusable. The official cleaning and reconciliation are implemented only in `01`.

**Limit.** These checks establish consistency under stated rules, not complete
histories, representative market coverage, absence of bots or truthful reviews.

### E2. Which categories could sustain the project?

**Question and method.** Compare all six categories using observed activity, sample
support, prices and concentration; then compare LG with Samsung within each category.

**Result.** TVs had the most cleaned events (211,958), followed by washing machines
(161,871), vacuum cleaners (117,983), refrigerators (108,259), microwaves (26,569)
and monitors (21,718). Every category contained six retained brands. November had
more events in every category, motivating time adjustment without identifying
promotions or a seasonal cause. The historical all-category build yielded 363,277
viewed considerations.

TVs supplied the largest combined LG/Samsung consideration and purchase sample:
74,712 rows, 196 products and 2,150 purchased considerations. Washing machines were
a serious alternative: LG had 544 purchases versus 505 in TVs, although the combined
sample was smaller. Monitors offered close prices but only 13 observed LG purchase
outcomes. LG microwave activity was concentrated in its five leading models (89.4%);
LG refrigerator and vacuum-cleaner concentration was about 60%, versus 42.3% in TVs.
The washing-machine top-five share was 51.2%, and monitor share 31.9%.

**Conclusion and decision.** Retain TVs for the joint support/comparator/focus case;
do not claim they maximise every criterion. Washing machines were a viable route,
monitors had limited purchase support, and other categories brought different
support, concentration and price trade-offs. LG refrigerators had a higher median
price than LG TVs, so 'TVs are LG's most expensive category' was not a justification.
The all-category gap results were exploratory, not a significance contest.

**Limit.** A category's event volume is neither conversion nor independent sample
size. Product specifications and availability remain unobserved.


The current support and competitor tables are computed in Sections 4.1–4.2.


### E3. Why Samsung rather than the other competitors?

**Question and method.** Screen all 30 observed LG/competitor category pairs by
support, median first-view-price distance, central-price-range overlap and concentration.
The historical screen weighted considerations, not products equally.

**Result.** Samsung was closest in median price in five categories; Tefal was closer
for vacuum cleaners. Within TVs, Samsung had the closest median to LG (493.96
versus 460.84, a 7.19% distance), the only positive overlap of the central price
ranges, and the broadest competitor assortment (91 products). LG had 105 products;
Samsung supplied 52,499 considerations versus LG's 22,213. Artel, Haier, Kivi and
TCL were observed alternatives but less close in this price-positioning screen.

**Conclusion and decision.** Select Samsung as a named, feasible comparator and
retain the other brands as context and browsing history. Pooling all rivals changes
the price/product composition of the benchmark and is less specific managerially.
Category choice and competitor choice need separate arguments.

**Limit.** Neither price proximity nor central-range overlap matches specifications,
stock or shopper intent. The catalogue matrix counts distinct products, whereas the price screen weights
considerations. Current detailed TV price distributions belong to `02`.

The current support and competitor tables are computed in Sections 4.1–4.2.


### E4. Was satisfaction a genuine alternative?

**Question and method.** Inspect review support by brand/category/date, star ratings,
verification and price/text availability rather than reject the file wholesale.

**Result.** Refrigerators/freezers and laundry contained 5,834 and 5,306 reviews;
cooking had 4,113 reviews and the largest product variety (827 products), dishwasher
1,959, other appliances 2,759, and air conditioners only 33 reviews across five
products. LG and Samsung overlapped in cooking, dishwashers, laundry and
refrigerators/freezers. Samsung had 1,601 one-star versus 486 five-star reviews;
LG had 1,229 versus 1,026. Verified reviews numbered 414 of 2,684 for Samsung and
1,022 of 2,923 for LG. Both therefore had more unverified than verified reviews.

**Conclusion and decision.** A satisfaction/text project was feasible in overlapping
appliance categories, with category/time/product adjustment and a verified-only
sensitivity. It would answer a different managerial question, so it was not joined
to the TV conversion study. The review tables in Section 5 retain the full rating,
verification, coverage and candidate summaries independently of the retired file.

**Limit.** Review caps and self-selection prevent market-share or defect-rate claims.
Unverified is not synonymous with fake. Raw brand ratings do not establish an
adjusted quality difference or explain individual TV purchases.

### E5. What did the initial journey benchmark reveal?

**Question and method.** Start from viewed user/session/product considerations and
compare observed view→cart, eligible-cart→purchase and view→purchase rates, then
check month, price overlap and repetition.

**Result.** LG's historical rates were 4.83%, 34.86% and 2.27%, against Samsung's
6.31%, 40.17% and 3.13%. The gaps were -1.48, -5.31 and -0.86 percentage points.
There were 446 purchased considerations without a qualifying cart path, so multiplying
the two stage rates did not recover overall purchase conversion. There were 1,114
first views on the last observed day, motivating an observation-window check.

The stage gaps appeared in both months but changed size: LG's view→cart rate rose
from 3.10% to 5.59%, Samsung's from 5.69% to 6.64%; cart→purchase fell from 44.76%
to 32.44% for LG and from 46.25% to 37.33% for Samsung. Central prices overlapped
substantially. Repetition mattered: 15,561 users contributed multiple rows, accounting
for 72.82% of considerations. Five products generated 42.31% of LG rows and 52.38%
of Samsung rows.

**Conclusion and decision.** Price/time adjustment, explicit stage denominators,
observation-window checks and dependence-aware uncertainty were needed before
interpreting a brand gap. The funnel became a diagnostic; the recent primary RQ
places it inside the broader purchased-count shortfall rather than equating conversion
with the whole business problem.

**Limit.** A zero is no qualifying observed transition in that unit; it is not proof
of no eventual purchase. The denominator changes after cart entry; a cart-stage
gap does not identify checkout friction. `02` owns current funnel/period exhibits.
<details>
<summary>Historical monthly stage results</summary>

| Brand | Month | Considerations | Carts | Purchases after cart | View→cart (%) | Cart→purchase (%) |
| --- | --- | --- | --- | --- | --- | --- |
| lg | 2019-10 | 6,770 | 210 | 94 | 3.102 | 44.762 |
| lg | 2019-11 | 15,443 | 863 | 280 | 5.588 | 32.445 |
| samsung | 2019-10 | 18,494 | 1,053 | 487 | 5.694 | 46.249 |
| samsung | 2019-11 | 34,005 | 2,258 | 843 | 6.640 | 37.334 |

</details>

### E6. Did popular products or the shopper selection change the picture?

**Question and method.** Remove each brand's single most-considered product, and
separately restrict to users who considered both brands during the two months.

**Result.** Removing LG P000640 (2,723 considerations) and Samsung P000507 (9,106)
left 19,490 and 43,393 rows. View→cart became 4.64% versus 5.61%, and cart→purchase
34.81% versus 38.83%: the gaps narrowed, but the assortment also changed.
The whole-period dual-brand group contained 6,364 users and 28,027 rows. Its
view→purchase rates were 1.386% for LG and 1.421% for Samsung, a gap of about
-0.035 pp. View→cart differed by -0.042 pp; cart→purchase was +0.859 pp for LG.
The dual-brand category comparisons are preserved in E10's supporting results.

**Conclusion and decision.** Aggregate comparisons were sensitive to assortment
and selected audiences. Investigate concentration and observed shopper mix rather
than attribute the raw gap directly to pages or checkout. The full sample remained
the reference; restricted samples were sensitivities.

**Limit.** These are the same observed users in both brand denominators, not matched
products/times/intent or a within-person experiment. Whole-period membership uses
future information relative to early views. Near-zero estimates do not establish
equivalence. Product exclusions answer a narrower question.

### E7. What were the shopper types, and why did mix matter?

**Question and method.** Label users LG-only, Samsung-only or both-brand over the
full observation window; compare group rates and descriptive user profiles.

**Result.** There were 6,544 LG-only users, 22,963 Samsung-only users and 6,364
cross-shoppers. Cross-shoppers contributed 54.90% of LG considerations but 30.16%
of Samsung considerations. Their per-consideration purchase rates were about
1.39%/1.42%, below the single-brand rates of 3.35% for LG-only and 3.87% for Samsung-only.
Cross-shoppers considered a mean of 4.40 TVs, versus 1.53/1.60 in the single-brand
groups; they also covered more products, brands and sessions. Their probability of
any recorded TV purchase was 6.32%, versus 5.41%/6.41%: lower conversion per row
was not equivalent to a uniquely low purchase propensity per user.

**Conclusion and decision.** Row-denominator dilution and mix were credible reasons
to examine browsing state before interpreting an average conversion difference.
Use user-level and consideration-level summaries for their respective questions;
do not turn these behavioural labels into demographic or intent segments.

**Limit.** Whole-period profiles are retrospective; missing sampled sessions can
change membership. Different groups do not identify the mechanism behind purchase.
<details>
<summary>Historical group KPIs and complete user-profile summary</summary>

| Group | Brand | Users | Considerations | View→cart (%) | Cart→purchase (%) | View→purchase (%) |
| --- | --- | --- | --- | --- | --- | --- |
| lg_only | lg | 6,544 | 10,019 | 6.318 | 37.441 | 3.354 |
| samsung_only | samsung | 22,963 | 36,666 | 7.454 | 42.261 | 3.873 |
| cross_shopper | lg | 6,364 | 12,194 | 3.608 | 31.136 | 1.386 |
| cross_shopper | samsung | 6,364 | 15,833 | 3.651 | 30.277 | 1.421 |

| shopper_type | users | median_tv_considerations | mean_tv_considerations | mean_tv_sessions | mean_distinct_tv_products | mean_tv_brands_in_recorded_events_all_6 | pct_browsed_other_categories | median_price_viewed | pct_users_with_a_purchase_here | pct_users_bought_any_tv |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| lg_only | 6,544 | 1.000 | 1.531 | 1.084 | 1.473 | 1.275 | 11.293 | 450.200 | 4.630 | 5.410 |
| samsung_only | 22,963 | 1.000 | 1.597 | 1.078 | 1.548 | 1.292 | 10.286 | 488.800 | 5.766 | 6.406 |
| cross_shopper | 6,364 | 3.000 | 4.404 | 1.343 | 4.202 | 2.635 | 13.828 | 468.962 | 5.374 | 6.317 |

</details>

The historical all-six-brand count uses all recorded TV event types, including
cart and purchase events. It is not restricted to brands with a recorded view.


### E8. What did reweighting and the shopper-type Logits change?

**Question and method.** Apply Samsung's group weights to LG's group-specific rates;
then compare price/month Logits with a cross-shopper indicator and a brand×group
interaction, separately for three outcomes. For cart→purchase, use cart shares.

**Result.** Direct reweighting changed the purchase gap from -0.860 to -0.373 pp,
a 56.60% accounting contribution under that reference weighting. The corresponding
contributions were 45.40% for view→cart and 27.94% for cart→purchase.
Adding retrospective cross-shopper status to the price/month Logit moved the
standardised purchase gap from -0.827 to -0.281 pp; view→cart moved from -1.563 to
-0.772 pp and cart→purchase from -4.161 to -1.878 pp. With an interaction, purchase
gaps were -0.448 pp among single-brand users and -0.049 pp among cross-shoppers;
the purchase interaction p-value was 0.461. The view→cart interaction p-value was
0.033 and cart→purchase 0.177, under the old user-only covariance convention.

**Conclusion and decision.** The baseline estimate depended materially on how
observed browsing composition was represented. This motivated a prior-only exposure
check, not a declaration that a measured percentage of lost purchases was caused
by audience mix. The newer within-session mix analysis in `02` uses a different
classification; its shares must not be substituted for these whole-period results.

**Limit.** Weights are reference-dependent; the groups include future behaviour.
The model contrasts lacked probability-gap intervals and user-only clustering did
not jointly handle product dependence. Do not infer equivalence from nonsignificance.
<details>
<summary>Historical reweighting and all three model-sequence outcomes</summary>

| kpi | LG_raw_pct | LG_with_Samsung_mix_pct | Samsung_pct | raw_gap_pp | gap_after_mix_pp | share_of_gap_explained_by_mix_pct |
| --- | --- | --- | --- | --- | --- | --- |
| view_to_cart_pct | 4.831 | 5.501 | 6.307 | -1.476 | -0.806 | 45.404 |
| cart_to_purchase_pct | 34.856 | 36.340 | 40.169 | -5.314 | -3.829 | 27.941 |
| view_to_purchase_pct | 2.273 | 2.760 | 3.133 | -0.860 | -0.373 | 56.602 |

| outcome | N | positives | M1_adjusted_gap_pp | M2_adjusted_gap_pp | M3_gap_single_brand_pp | M3_gap_cross_shoppers_pp | M3_interaction_p | all_converged |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| view_to_purchase (Y) | 74,712 | 2,150 | -0.827 | -0.281 | -0.448 | -0.049 | 0.461 | 1 |
| view_to_cart | 74,712 | 4,384 | -1.563 | -0.772 | -1.275 | -0.111 | 0.033 | 1 |
| cart_to_purchase | 4,384 | 1,704 | -4.161 | -1.878 | -3.534 | 1.563 | 0.177 | 1 |

</details>

### E9. Did the lesson survive using only exposure before the first view?

**Question and method.** Replace future cross-shopper membership with whether the
other main brand had already been observed for the user strictly before that row's
first view, including earlier recorded sessions. Fit on the same stage samples.

**Result.** Earlier other-main-brand exposure was recorded for 41.63% of LG rows
versus 14.47% of Samsung rows. The standardised purchase gap moved from -0.827
to -0.486 pp; view→cart from -1.563 to -1.030 pp; cart→purchase from -4.161 to
-3.355 pp. All these fits converged.

**Conclusion and decision.** Browsing history deserved attention even when future
behaviour was excluded, but the attenuation was smaller than with whole-period
labels. The official build therefore prioritises predictors timed before the first
view and separates within-session history from incomplete cross-session history.
This supports including browsing in the analysis; it does not resolve which
managerial intervention would work or make the old stage RQ permanently preferable.

**Limit.** This is not identical to the current `other_brand_opened_before`, which
uses any other TV brand within the same session. Seven elapsed days do not recover
unsampled history; a zero means no earlier exposure recorded, not none experienced.
<details>
<summary>Historical prior-exposure model results</summary>

| outcome | M1_adjusted_gap_pp | M2_prior_exposure_gap_pp | prior_exposure_odds_ratio | converged |
| --- | --- | --- | --- | --- |
| view_to_purchase (Y) | -0.827 | -0.486 | 0.569 | 1 |
| view_to_cart | -1.563 | -1.030 | 0.649 | 1 |
| cart_to_purchase | -4.161 | -3.355 | 0.849 | 1 |

</details>

### E10. What did the broader category and model experiments add?

**Question and method.** Compare LG against Samsung and pooled rivals in every
category, repeat the comparison among dual-brand users, and on TVs add pre-view
browsing counts, prior sessions, a user conditional Logit and price-quintile interactions.

**Result.** The raw purchase gap against Samsung was -0.675 pp for microwaves,
-0.596 for monitors, +0.134 for refrigerators, -0.860 for TVs, -1.041 for vacuum
cleaners and -0.307 for washing machines. Among selected dual-brand users it became
+0.124, +0.166, -0.227, -0.035, +0.113 and +0.261 pp respectively: audience
restriction could change the sign, not just the magnitude. TVs were not the largest
raw-gap category. Against pooled non-LG rivals, LG's refrigerator gap was negative
even though it was positive against Samsung, illustrating comparator sensitivity.

For TVs, the model sequence moved from -0.860 pp (brand only) to -0.827 pp
(price/month) to -0.515 pp with pre-view covariates. LG rows had more products
opened before them (2.023 versus 1.346), more prior brands (1.010 versus 0.728),
more prior sessions (0.402 versus 0.224) and more within-session rival-main-brand
exposure (36.3% versus 11.6%). The last flag differs from E9's across-session
prior-exposure definition. The selected user conditional Logit used 1,697 rows
and gave LG OR 0.944, p=0.618, with its model-based uncertainty. The price-quintile
experiment found lower raw LG conversion in every quintile; individual interaction
p-values ranged from 0.112 to 0.736, without establishing homogeneous responses.
No joint interaction test or causal price interpretation is inferred here. Only **332 dual-brand users with both purchase outcomes** supplied
the selected user conditional Logit. Related stage samples had 803 users with
varying cart outcomes and 36 with varying cart→purchase outcomes.

**Conclusion and decision.** Retain the broad screen as a check against arbitrary
scope, take pre-view history seriously, and use fixed effects or price interactions
only to address a stated concern. A conditional user model estimates a selected
within-user association; it does not eliminate brand-specific preferences, and its
nonsignificance cannot establish that LG and Samsung convert equivalently.

The user conditional Logit uses model-based errors; the other historical Logits
use user-clustered errors.

**Limit.** The original category z-statistics treated rows as independent and are
not valid final significance rankings. The historical price-quintile model was
exploratory and used user-only clustering. No specifications, causal price variation
or complete histories were recovered. These experiments did not determine a final
model or establish an optimal pricing action.

**Definitions of the historical controls.** `pre_products` and `pre_brands` count
distinct products and brands in view events from **all categories**, strictly before
the consideration's first view in the same user-session. `other_brand_tv` flags
an earlier LG/Samsung rival TV view in that session. `prior_sessions` counts the
user's recorded sessions starting before the current session, across all categories.
These differ from `01`'s TV-only prior counts and binary returning-visitor flag.

Price quintiles were formed with `pd.qcut(initial_price, 5, labels=False)` on the
pooled LG/Samsung TV considerations, weighted by consideration rather than equally
by product. They were sample-relative exploratory bins, not `02`'s fixed price bands.
Detailed historical coefficient snapshots remain archival metadata; the concise
results above retain the lessons without duplicating the current modelling step.


### E11. Which initial stage models were methodological starting points?

**Question and method.** The first stage Logits used initial log price and month,
with month defined at first view for view→cart and at eligible cart for cart→purchase.

**Result.** Standardised view→cart probabilities were 4.774% for LG and 6.338% for
Samsung, a -1.563 pp gap. Cart→purchase probabilities were 35.718% and 39.879%, a
-4.161 pp gap. Both models converged. They established feasibility, not a complete
explanation: adding browsing history subsequently changed both contrasts, as E8–E9
show. There were no confidence intervals for these probability differences.

**Conclusion and decision.** Preserve the binary-response methodology, explicit
stage populations and probability summaries; rebuild substantive models on current
variables with appropriate uncertainty. The supplied `03` now uses LPMs with Logit
checks for conversion, selected-visit choice modelling and a separate count model.
It is not necessary to keep a second official implementation of these old fits.

**Limit.** Coefficient intervals are not probability-gap intervals; user-only
clustering leaves product dependence unresolved. Separate stage probabilities do
not identify causes of abandonment or multiply to all purchases.

### E12. How did the decisions follow from the complete record?

1. **One coherent source:** usable behavioural journeys answered the selected
   retailer question; review coverage supported a different satisfaction project.
2. **A defensible scope:** broad category/comparator screens supported TVs and
   Samsung while acknowledging washing machines and other alternatives.
3. **One measurement contract:** repetitions, shared session IDs, timing and
   concentration required the explicit datasets and audit now implemented by `01`.
4. **Description before explanation:** raw stage differences, sensitivity to products
   and dual-brand users made the shopper state and assortment impossible to ignore;
   `02` owns the current descriptive evidence, including the reach/conversion identity.
5. **A broader primary RQ:** the old stage comparison remained diagnostic; the recent
   direction distinguishes purchase-count shortfall, brand reach, browsing intensity
   and conversion. The selected `03` addresses corresponding adjusted associations.
6. **An appropriately limited decision:** patterns prioritise investigation and
   retailer-supported testing; no analysis here identifies broken pages, causal
   audience effects, optimal discounts or a budget allocation.

**Coverage boundary.** All substantive routes and their decision-relevant outcomes
are described here; aggregate supporting results are embedded. The in-memory scope screen does not export or supply the official datasets.
The next section identifies the maintained owner of each analytical task. Failed reasoning, outdated
measurement and provisional p-values are explained rather than silently promoted
to final findings. Reading this record requires no old code.


## 7. What the maintained workflow owns

The four exploratory implementations have been consolidated conceptually into this
workflow. They are not execution or reading dependencies. This map is organised by
analytical task so the maintained project can be understood without the earlier files.

| Analytical task | Retained evidence or rationale | Maintained calculation |
| --- | --- | --- |
| Source meaning, schema and basic quality | Section 3; E1 | `00` read-only audit; `01` official event cleaning |
| Category and competitor choice | Section 4; E2–E3 | `00` recomputes support, first-view prices and concentration for every candidate |
| Review-based alternative | Section 5; E4 | `00` recomputes review quality, coverage, dates, ratings and verification |
| TV measurement, eligible outcomes and prior browsing | Section 8; E5 | `01` builds and audits official TV datasets |
| Funnel, periods, price overlap and concentration | E5–E6 | `02` current descriptive exhibits; `03` targeted model sensitivities |
| Shopper composition and prior exposure | E7–E9 | `02` current composition analysis; `03` observed-history adjustments |
| Early stage Logits, selected-user models and price-quintile experiments | E8–E11 historical lessons | Superseded specifications, not required fits; current models belong to `03` |
| Reach, breadth of browsing and conversion | Sections 1 and 9 | `02` accounting decomposition; `03` adjusted associations on explicitly distinct endpoints |

A historical result is not a target that current models must reproduce. Earlier
strict-timestamp outcomes, retrospective shopper groups and user-only inference
are labelled in E1–E12. Current scope screening follows the first-view timing rule
of `01`; official final conclusions use `02`–`03` and their uncertainty conventions.
Removing old implementations does not make the exploratory scope choice prespecified.


## 8. Measurement choices and the official build

| Choice | Why it follows from the exploration | Implementation boundary |
| --- | --- | --- |
| User × session × product with a view | Views can repeat; session IDs are not guaranteed to identify a user; a product consideration is not a person or a unit sold | `00` uses this key for scope screening; `01` builds and checks the official TV dataset |
| First view sets price and time | Predictors must describe the situation before the outcome | `01` saves `first_view`, `price` and `month` |
| Purchase in the same consideration/session | A clear observed window avoids inventing full customer histories | `01` constructs `purchased`; zero means no qualifying observed outcome |
| Same-second inclusion | Coarse timestamps cannot resolve every order; the current pipeline has an explicit rule | `01` audit and `02` measurement discussion; any alternative is a labelled later sensitivity |
| Retain all six TV brands for history | Other brands can be part of a shopper's path before LG or Samsung | `01` constructs `other_four` and the prior-brand flags |
| Whole-session prior browsing | History before the first view describes the ongoing search; restarting after purchases changes its meaning | Whole-session primary definition in `01`; reset sensitivity and results owned by `01`/`02` |
| Limit cross-session history claims | Whole sessions were sampled, not full user histories | Returning/prior-purchase flags describe recorded evidence, not new-customer or non-buyer status |
| Top models as a retrospective split | Concentration can make aggregate brand comparisons sensitive to the assortment | `01` defines and audits the rule; `02` reports results; `top_model` is not a first-view shopper characteristic |

Controls are justified by timing and the comparison they clarify, not selected just
because their raw conversion differences are large. Later cart entry, purchases,
`session_result`, eventual brand totals and whole-period shopper types must not be
used as if they were known at the first view in the main purchase model.


## 9. The analytical sequence after the scope decision

### 9.1 Distinguish purchase counts, reach and conversion

For brand *b*, let **S** be distinct user-sessions opening at least one model of that
brand, **C** the viewed considerations and **P** the purchased considerations. Then:

**P = S × (C / S) × (P / C)**.

This is an accounting identity: sessions reaching a brand, considerations per
reaching session, and purchase conversion. It motivates the organisation of `02`,
not a new model to fit in `00`. The factors and their contributions are calculated
only in `02`, Section 4.7. A session can reach both brands; reach counts are not
disjoint audiences, unique people or ad impressions. Purchased considerations
are not verified physical units, revenue or profit.

The identity separates a count difference from a rate difference. Its contributions
are descriptive accounting, not recoverable purchases, causal effects or recommended
budget shares. The prior decision to study TV/LG–Samsung is justified by the scope
evidence, not retrospectively by whichever decomposition share is largest.

### 9.2 Describe before estimating

`02` should establish the observed pattern and inspect prices, funnel denominators,
shopper state, navigation and concentration before substantive models are interpreted.
Its raw-gap regressions and uncertainty calculations are inferential support for
descriptive comparisons, not a completed adjusted explanation. This notebook makes
no claim that a particular gap persists, disappears or is equivalent to zero.

### 9.3 Step 03: modelling conversion and considerations per product

The executable `03_models.ipynb`, headed **Models for Section 5: LG and
Samsung TVs**, implements the modelling step. It follows `02` conceptually and reads
`tv_considerations.csv`, `tv_considerations_top_models.csv` and `tv_events_clean.csv`
from `01`. It does not need to rerun the scope screening or copy the descriptive
exhibits. It saves Tables 5.1–5.8, Figure 5.1 and an exhibit index under `results/models/`.

| Block in 03 | Unit and sample | Purpose and boundary |
| --- | --- | --- |
| **Model 1: conversion** | LG/Samsung considerations | Linear probability models with raw, position/price/week-adjusted and richer pre-view-history versions; top-model interaction; Gelbach decomposition of coefficient changes; targeted sensitivities, including purchase Logit average discrete contrasts. These are observed-covariate comparisons, not repeated observations of an identical shopper under alternative brands. |
| **Model 2: within-visit choice** | TVs first opened at or before the qualifying purchase, in visits retaining at least two alternatives and exactly one purchased consideration; visits with a purchase of another product are excluded; all six brands, plus an LG/Samsung head-to-head subset | Conditional Logit with visit fixed effects and a within-visit LPM check. This concerns which opened model was purchased conditional on the selected visit ending in one purchase, not unconditional conversion. |
| **Model 3: considerations per model** | One observation per viewed LG/Samsung product | Poisson conditional-mean regression with robust errors; alternative price controls, top-model exclusion, an observed-span offset and log-count OLS sensitivity. This concerns product-level consideration counts, not directly the brand-level number of sessions reaching any of its models. |
| **Model 4: optional price extension** | Considerations for conversion; product-day/week panels for counts | Product and time fixed effects examine price associations within a product. Prices on unobserved periods and the unobserved listing/availability window need explicit sensitivity checks; no causal elasticity or recommended discount follows. |

This records the implemented scope; inferential assumptions and interpretation still require the qualifications in `03`. The earlier purchase-Logit sequence was a methodological starting
point; it is not a separate pipeline to run in addition to `03`. The notebook uses
an LPM as the conversion baseline and Logit as a robustness check. A purchase Logit
still connects the binary outcome to the established course method. More advanced
blocks require a specific purpose and confirmation that they fit the course's
method requirements; no need to keep all extensions in the final report.

The baseline conversion controls are position, log price
and calendar-week effects. Its richer version replaces position with grouped prior
TV counts, rival/own-brand and other-four prior-view flags, plus returning visitor.
Keep the same sample when comparing specifications. Report counts, overlap, sparse
cells, convergence, LPM predictions outside [0,1] and uncertainty. `top_model` uses
the whole period and is a retrospective split, not a prospectively known shopper
characteristic; the notebook explicitly distinguishes that exception from the pre-view conversion controls. Product-level median prices and panel-period prices are retrospective summaries too.

Do not include redundant encodings of the same history. Brand-relative flags need
careful prediction design: a standardised profile must not combine inconsistent
absolute and relative histories. Differences in a coefficient or in a predicted
gap are conditional associations, not a causal explained share. The LPM/Gelbach
identity and the Logit probability contrast answer related but distinct questions.


### 9.4 Report probabilities with uncertainty, not coefficient changes alone

For the LPM baseline, a coefficient on brand gives a percentage-point difference
under its specified additive structure; inspect its probability-range limitations.
For the Logit checks, on a stated common population average the predicted probabilities for
the LG and Samsung profiles and report their paired difference in **percentage
points**. Report the odds ratio as a separate technical summary. In nonlinear
models, changes in log-odds coefficients across specifications are not an explained
share; even changes in standardised gaps are conditional associations.

The probability difference needs its **own** confidence interval from a justified
covariance/delta-method calculation or appropriate resampling. Coefficient intervals
are not probability-gap intervals. Earlier notebooks omitted gap intervals and
clustered only by user. Current `02` explicitly recognises user and product
dependence; later modelling should carry that concern forward with a documented
joint treatment, rather than silently returning to independent rows. Product-cluster
or small-cluster sensitivity is particularly important in restricted top-model
samples. The large number of considerations does not create many independent
brand assignments: brand is constant within product.

Do not add product fixed effects while expecting a separately identified brand
coefficient. No significant difference is not evidence of equivalence. Any
equivalence claim would require its own meaningful margin and appropriate test.

### 9.5 Secondary models and checks: one concern per addition

| Concern | Candidate check | Why it remains secondary |
| --- | --- | --- |
| Is the overall comparison sensitive to browsing history? | M2 versus M3 on the same population; alternative prior-view encodings | Incomplete observed history and remaining omitted variables prevent a causal explanation |
| Do the funnel diagnostics warrant a stage model? | Logit for `added_to_cart`; Logit for `purchased_after_cart` among eligible carts | Selected cart denominator; entry month should correspond to the relevant stage and may require rebuilding a cart timestamp not saved in the current CSV |
| Are aggregate differences sensitive to a few products? | Full versus prelabelled top/non-top or leave-one-product-out checks | Changes the assortment and independent product support; `02` owns descriptive split results |
| Could stable shopper differences matter? | Selected dual-brand users; conditional user Logit only with within-user outcome and covariate variation | Selected estimand; loses invariant outcomes; time-varying intent and brand-specific preferences remain |
| Is the price adjustment too rigid? | A prespecified nonlinear price form or a limited interaction with adequate overlap | Listed price is not randomly assigned; a price interaction is not elasticity or a discount recommendation |
| Does the observation rule matter? | Same-second, end-of-window and prior-purchase sensitivities | Changes what is observed or who is included; report changed denominators |
| Is the chosen link driving the average contrast? | A labelled Logit/LPM comparison on the same sample | Not a replacement for Logit diagnostics or dependence-aware uncertainty |

Whole-period cross-shopper profiles and their reweighting supplied the useful
warning that considering many models can reduce a per-consideration rate without
lowering a user's probability of any purchase. They do not demonstrate that shopper
mix causes the gap. Restricting to people who viewed both brands does not match
products, timing, intent or number of rows contributed by those people.

For clarity, the original descriptive labels mean **LG-only**, **Samsung-only** or
**both brands observed for that user over the full window**. They are not lifetime
loyalty segments. A user profile can count considered products, sessions, distinct
TV brands, other categories visited and whether any TV purchase was recorded;
none is a measured intention or demographic. Whole-period totals include future
information relative to earlier views. A shopper considering ten models and buying
one has one purchase across ten considerations but still one user with a purchase:
these denominators answer different questions.

The descriptive mix calculation has a precise interpretation. If `w_bg` is brand
`b`'s share of considerations in group `g` and `r_bg` is its within-group purchase
rate, its rate is `sum_g(w_bg * r_bg)`. Applying Samsung's weights to LG's group rates
gives `sum_g(w_Sg * r_Lg)`. Subtracting and adding this quantity splits the raw gap
into a composition term and a within-group term. The split depends on the reference
weights; it is not a causal attribution. For cart-to-purchase the weights must be
shares of eligible carts, not shares of all views. `02` owns the current descriptive
reweighting using first-view browsing state; there is no second implementation here.

The prospective alternative is **other brand observed strictly before this first
view**. Within-session history is primary; earlier-session exposure can miss
unsampled visits. A narrower observed history is not proof of an absence of earlier
exposure. A difference between retrospective and prior-only analyses diagnoses
definition sensitivity, not which visitor group an intervention can profitably target.

Separate funnel probabilities cannot generally be multiplied to reconstruct overall
purchase conversion because recorded purchases may lack an eligible cart path.
A persistent stage gap motivates investigation; it does not diagnose defective
pages, payment or delivery. Stop adding techniques once they no longer resolve a
stated uncertainty. PCA lacks a defensible measured-attribute set for this question;
multinomial choice modelling requires a defined choice outcome and choice set, not
the current priority-based consideration label.

## 10. Questions the rationale must withstand

| Likely question | Answer and evidence boundary |
| --- | --- |
| Why not use both supplied datasets? | Different products/users and no valid join; review feasibility is documented in Section 5. Two datasets are not intrinsically better than one coherent dataset. |
| Why TVs instead of washing machines? | A focused, supported two-brand comparison with a feasible benchmark; washing machines remain credible. `02`, Table 4.1 gives current support; no claim that TVs win every criterion. |
| Why Samsung? | The original all-competitor screen combined support, assortment and price proximity. Current TV price evidence is in `02`, Section 4.2. Similar prices do not equal comparable specifications. |
| Was scope chosen after looking at the data? | Yes. It is an exploratory, disclosed scope decision, not a preregistered test. All-category results cannot be used as independent confirmation of a category selected from them. |
| Why has the old funnel RQ evolved? | A purchase-count shortfall can reflect reach and browsing intensity as well as conversion. Funnel stages remain useful diagnostics rather than competing primary endpoints. |
| Why build 01 rather than use an old notebook's table? | One explicit current measurement contract, reproducible variables and audit trail; historical timing and history definitions differ. |
| Why is 02 needed before Logit? | It establishes the observed patterns, support and sensitivities a model must address; it prevents interpreting an adjusted coefficient without understanding its denominator. |
| Does a narrowing adjusted gap prove an audience problem? | No. It shows sensitivity to measured composition on a stated population; unobserved intent, specifications and selection remain. |
| Does first-brand order identify awareness or paid placement? | No. No ad exposure or placement is observed; even a random-order benchmark conditions on the models eventually viewed. |
| Can the findings tell LG where to spend its budget? | They can identify questions and candidates for a retailer-supported test. Returns and optimal allocation require incremental effects, costs and margins absent here. |
| Are non-top models equally good when a test is nonsignificant? | Not established. Weighting, product selection and uncertainty matter; no equivalence claim follows from a large p-value. |

The team's assessed report must present its own reasoning and disclose AI support.
This working notebook is not text to paste into the final report.

## 11. Reproducibility and handoff

Run all code cells from a fresh Python session in the notebook directory. This
notebook needs only the two original CSVs and `DATA_DICTIONARY.md` in the sibling
data directory. The next cell fingerprints those inputs and records package versions.
No other project file is required, imported, executed or written.

The recomputed screening tables are derived from those sources in memory. They do
not replace the official cleaning, history variables or exported files of `01`.
The selected historical results in E1–E12 and the metadata archive are fixed records
of exploratory work, not a promise that running `00` re-estimates those models.
Their source fingerprints and original reproduction date are retained as provenance.

This notebook establishes why the scope is defensible. Current report figures,
estimates and recommendations must follow the definitions and uncertainty reported
by the maintained `01`–`03` workflow.


In [9]:
# Fingerprint the only runtime inputs; record the screening environment.
import platform
import IPython
source_versions = []
for name in required:
    path = DATA_DIR / name
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    source_versions.append({'Source': str(path), 'SHA-256': digest.hexdigest()})
display(pd.DataFrame(source_versions).set_index('Source'))
display(pd.Series({'Python': platform.python_version(), 'pandas': pd.__version__,
                   'numpy': np.__version__, 'IPython': IPython.__version__}, name='Version').to_frame())


,SHA-256
Source,
../data/01_LG_Electronics_Appliances/electronics_market_behavior.csv,3303a56e62cff3e38491d58e41f40706317c500039c26f135bc89a811bc59022
../data/01_LG_Electronics_Appliances/appliance_brand_reviews.csv,64b57e87b44ec047403bf1b29bdc60ecd9ead2add64b8327ac6011524bb31a60
../data/01_LG_Electronics_Appliances/DATA_DICTIONARY.md,85d0b484b7a95abfcbdb49efbd78c21a9589470da9829d8850df8c6b52a3e8d6


,Version
Python,3.14.0
pandas,3.0.1
numpy,2.3.4
IPython,9.7.0


**Handoff:** `00` explains the route and its limits. Run `01_build_tv_dataset.py`
to implement the measurement decisions, then `02_preliminary_analysis.py` to
produce the descriptive evidence. The executable **`03_models.ipynb`** then addresses adjusted conversion, conditional
within-visit choice and product-level consideration counts, with price changes as
an optional extension. It reads the `01` datasets. Its estimates and conclusions
belong in that step, not in `00`. 
No old exploratory finding is a required result to recover.
